# P19 · Estructura de la red programada de correspondencias Metro–bus

> **¿Qué servicios conectan más estaciones y cómo cambia la red programada al retirarlos?**

Resultado reproducible de P03/P10 basado en GTFS DTPM y lugares bip! públicos.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/red_correspondencias_dtpm/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = []
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

Una estación con muchos servicios cercanos y un servicio que conecta muchas estaciones
son dos grados distintos. P19 modela ambos sin convertir la estructura programada en
una afirmación sobre pasajeros, puntualidad o capacidad.


## 2 · La solución que se va a aplicar

Se transforma cada lista de servicios de P03/P10 en aristas únicas, se calculan grados
en ambos lados y se retiran servicios por grado descendente en cinco escenarios. Cada
resultado puede reconstruirse desde la tabla de aristas.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Comparar el grado de los quince servicios programados que conectan más estaciones.
- **Fuente que leen las tres implementaciones.**
  [`resultados/servicios_centrales.csv`](resultados/servicios_centrales.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** grado de servicio en grafo bipartito. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** grado(s) = #{estaciones distintas e para las que existe la arista (e,s)}; cada par se cuenta una sola vez.
- **Interpretación.** El grado mide alcance estructural bajo el radio de P10; no mide pasajeros, frecuencia, capacidad ni puntualidad.
- **Error frecuente.** Contar filas antes de deduplicar pares estación–servicio puede inflar el grado.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `servicio` | `str` | categoría/eje X |
| `estaciones_conectadas` | `int64` | medida/eje Y |
| `participacion_estaciones_pct` | `float64` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso red_correspondencias_dtpm.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "red_correspondencias_dtpm" / "resultados" / "servicios_centrales.csv"
REQUIRED = {"servicio", "estaciones_conectadas"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.sort_values(["estaciones_conectadas", "servicio"], ascending=[False, True])[["servicio", "estaciones_conectadas"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["servicio"].tolist()
    y = resultado["estaciones_conectadas"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.invert_yaxis()  # el primer puesto del ranking queda arriba, como se lee una tabla
    ax.set_xlabel("Estaciones conectadas (recuento)")
    ax.set_ylabel("Servicio")
    ax.set_xlim(left=0)
    ax.set_title("Servicios programados con mayor alcance sobre estaciones de Metro")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Correspondencias reproducibles P03/P10 · Período: publicación GTFS analizada", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/red_correspondencias_dtpm/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/red_correspondencias_dtpm/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso red_correspondencias_dtpm.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "red_correspondencias_dtpm" / "resultados" / "servicios_centrales.csv"
REQUIRED = {"servicio", "estaciones_conectadas"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.sort(["estaciones_conectadas", "servicio"], descending=[True, False]).select(["servicio", "estaciones_conectadas"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["servicio"].to_list()
    y = resultado["estaciones_conectadas"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.invert_yaxis()  # el primer puesto del ranking queda arriba, como se lee una tabla
    ax.set_xlabel("Estaciones conectadas (recuento)")
    ax.set_ylabel("Servicio")
    ax.set_xlim(left=0)
    ax.set_title("Servicios programados con mayor alcance sobre estaciones de Metro")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Correspondencias reproducibles P03/P10 · Período: publicación GTFS analizada",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/red_correspondencias_dtpm/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/red_correspondencias_dtpm/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso red_correspondencias_dtpm.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "red_correspondencias_dtpm", "resultados", "servicios_centrales.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("servicio", "estaciones_conectadas")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::arrange(dplyr::desc(.data$estaciones_conectadas), .data$servicio) |> dplyr::select(.data$servicio, .data$estaciones_conectadas)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["servicio"]], y = .data[["estaciones_conectadas"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::scale_x_discrete(limits = rev) +
    ggplot2::coord_flip() +
    ggplot2::labs(
      title = "Servicios programados con mayor alcance sobre estaciones de Metro",
      x = "Estaciones conectadas (recuento)", y = "Servicio",
      caption = "Fuente: Correspondencias reproducibles P03/P10 · Período: publicación GTFS analizada"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/red_correspondencias_dtpm/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.red_correspondencias_dtpm import solution as network

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Declarar los dos tipos de nodo


Estación y servicio no son intercambiables.

> ⚠️ Usar el nombre de estación como ID fusiona homónimos.


In [ ]:
estaciones = network.cargar()
estaciones[['stop_id', 'stop_name', 'servicios_distintos']]


### Paso 2 · Convertir listas en aristas


Una relación por fila hace auditable cualquier medida de red.

> ⚠️ Contar el texto separado por comas incluye espacios y duplicados.


In [ ]:
aristas = network.aristas(estaciones)
aristas


### Paso 3 · Calcular el grado de cada servicio


El grado cuenta estaciones distintas conectadas.

> ⚠️ Contar aristas sin deduplicar sobrestima servicios repetidos.


In [ ]:
resultados = network.analizar()
resultados['servicios_centrales']


### Paso 4 · Calcular el grado de cada estación


La medida inversa cuenta alternativas programadas cercanas.

> ⚠️ Más servicios no implica más frecuencia ni capacidad.


In [ ]:
resultados['grado_por_estacion']


### Paso 5 · Publicar la regla de retiro


Los escenarios retiran primero los servicios de mayor grado, con desempate por ID.

> ⚠️ Cambiar el criterio entre escenarios invalida la comparación.


In [ ]:
{'orden': 'estaciones_conectadas descendente, servicio ascendente', 'escenarios': [0, 1, 3, 5, 10]}


### Paso 6 · Recalcular cinco escenarios


Cada escenario vuelve a contar aristas y estaciones con alternativa.

> ⚠️ Aristas perdidas no son pasajeros afectados.


In [ ]:
resultados['escenarios']


### Paso 7 · Visualizar los servicios de mayor alcance


La gráfica muestra grado estructural con fuente y unidad.

> ⚠️ Una barra alta no demuestra importancia social.


In [ ]:
resultados['servicios_centrales'][['servicio', 'estaciones_conectadas']]


### Paso 8 · Escribir lo que el modelo no observa


Los límites definen la siguiente fuente necesaria.

> ⚠️ Llamar resiliencia operacional a este escenario excede la evidencia.


In [ ]:
{'no_observa': ['pasajeros', 'GPS', 'capacidad', 'puntualidad', 'secuencia completa'], 'sí_observa': 'correspondencia programada por proximidad'}


## 4 · El resultado

El modelo mide relaciones programadas; la operación real requiere demanda, GPS y capacidad.

## 5 · Qué queda aprendido

1. El grano define el grafo.
2. Grado no es demanda.
3. Escenarios requieren regla fija.
4. Las aristas preservan trazabilidad.
